# Data Formatting
## customer_id / postal_code の変換テーブル作成
長い hex 文字列を連番整数に変換する。

In [ ]:
%load_ext cudf.pandas

In [15]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
print(f"rows: {len(customers):,}  unique customer_id: {customers['customer_id'].nunique():,}")
customers.head(3)

rows: 1,371,980  unique customer_id: 1,371,980


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...


In [ ]:
# customer_id 変換テーブル
unique_ids = customers["customer_id"].unique()
id_map = pd.DataFrame({
    "customer_id": unique_ids,
    "customer_id_int": range(1, len(unique_ids) + 1)
})

id_map.to_csv(PROCESSED_DIR / "customer_id_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'customer_id_map.csv'}")
id_map.head(3)

In [ ]:
# postal_code 変換テーブル
unique_postal = customers["postal_code"].unique()
postal_map = pd.DataFrame({
    "postal_code": unique_postal,
    "postal_code_int": range(1, len(unique_postal) + 1)
})

postal_map.to_csv(PROCESSED_DIR / "postal_code_map.csv", index=False)
print(f"変換テーブル保存完了: {PROCESSED_DIR / 'postal_code_map.csv'}")
print(f"unique postal_code: {len(postal_map):,}")
postal_map.head(3)

In [ ]:
# customer_id と postal_code を両方整数に変換して保存
customers_converted = (
    customers
    .merge(id_map, on="customer_id", how="left")
    .merge(postal_map, on="postal_code", how="left")
    .drop(columns=["customer_id", "postal_code"])
    .rename(columns={"customer_id_int": "customer_id", "postal_code_int": "postal_code"})
)
cols = ["customer_id", "FN", "Active", "club_member_status", "fashion_news_frequency", "age", "postal_code"]
customers_converted = customers_converted[cols]

customers_converted.to_csv(PROCESSED_DIR / "customers_converted.csv", index=False)
print(f"変換済み customers 保存完了: {PROCESSED_DIR / 'customers_converted.csv'}")
customers_converted.head(3)

In [17]:
# transactions_train の customer_id を変換テーブルで整数に変換して保存
transactions = pd.read_csv(
    RAW_DIR / "transactions_train.csv",
    dtype={"article_id": str}
)
print(f"rows: {len(transactions):,}")

transactions_converted = (
    transactions
    .merge(id_map, on="customer_id", how="left")
    .drop(columns=["customer_id"])
    .rename(columns={"customer_id_int": "customer_id"})
)
cols = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]
transactions_converted = transactions_converted[cols]

transactions_converted.to_csv(PROCESSED_DIR / "transactions_converted.csv", index=False)
print(f"変換済み transactions 保存完了: {PROCESSED_DIR / 'transactions_converted.csv'}")
transactions_converted.head(3)

rows: 31,788,324
変換済み transactions 保存完了: ..\..\data\processed\transactions_converted.csv


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,3,0663713001,0.050831,2
1,2018-09-20,3,0541518023,0.030492,2
2,2018-09-20,8,0505221004,0.015237,2
